# 테이블 생성/데이터 입력 노트북

이 노트북은 다음 순서로 사용합니다.
1. `TABLE_NAME`, `SCHEMA`를 확인
2. 테이블 생성 셀 실행
3. `ROWS`에 검사 마스터 데이터 작성
4. 업서트 셀 실행
5. 결과 조회 셀 실행


# 검사 마스터 데이터

`test` 테이블은 검사 정의의 루트 테이블입니다.
`scalecondition`, `itemcondition`, `normcondition`, `item`, `scale`, `norm`, `choice`보다 먼저 등록해야 합니다.


In [1]:
# 1) 테이블 이름 + 스키마 설정
from pathlib import Path
import re
import sqlite3

# DB 경로: 검사 설계용 별도 DB
DB_PATH = Path(r"C:\Users\user\workspace\2.0-modular\docs\modular_test.db")
TABLE_NAME = 'test'

# type: INTEGER(정수) / REAL(실수) / TEXT / BLOB(바이너리) 중 하나 사용
# constraints 예시: 'PRIMARY KEY AUTOINCREMENT', 'NOT NULL', 'DEFAULT 0'
SCHEMA = [
    {'name': 'id', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'name', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'descript', 'type': 'TEXT', 'constraints': ''},
]
PRIMARY_KEY = ['id']

VALID_TYPES = {'INTEGER', 'REAL', 'TEXT', 'BLOB'}
identifier_re = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')

def validate_identifier(name: str) -> None:
    if not identifier_re.match(name):
        raise ValueError(f'잘못된 이름: {name} (영문/숫자/언더스코어만 허용, 숫자로 시작 불가)')

validate_identifier(TABLE_NAME)
seen = set()
for col in SCHEMA:
    col_name = col['name']
    col_type = col['type'].upper()
    validate_identifier(col_name)
    if col_name in seen:
        raise ValueError(f'중복 컬럼명: {col_name}')
    if col_type not in VALID_TYPES:
        raise ValueError(f'지원하지 않는 타입: {col_type}')
    seen.add(col_name)

print('설정 확인 완료')
print('DB_PATH =', DB_PATH)
print('TABLE_NAME =', TABLE_NAME)
print('SCHEMA =', SCHEMA)
print('PRIMARY_KEY =', PRIMARY_KEY)


설정 확인 완료
DB_PATH = C:\Users\user\workspace\2.0-modular\docs\modular_test.db
TABLE_NAME = test
SCHEMA = [{'name': 'id', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'name', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'descript', 'type': 'TEXT', 'constraints': ''}]
PRIMARY_KEY = ['id']


In [2]:
# 2) 테이블 생성
column_defs = []
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').strip()
    part = f'"{c_name}" {c_type}'
    if c_constraints:
        part += f' {c_constraints}'
    column_defs.append(part)

if PRIMARY_KEY:
    pk_cols = ', '.join([f'"{c}"' for c in PRIMARY_KEY])
    column_defs.append(f'PRIMARY KEY ({pk_cols})')

create_sql = f'CREATE TABLE IF NOT EXISTS "{TABLE_NAME}" (' + ', '.join(column_defs) + ')'
print('실행 SQL:', create_sql)

conn = sqlite3.connect(DB_PATH)
try:
    conn.execute(create_sql)
    conn.commit()
    print(f'테이블 생성 완료: {TABLE_NAME}')
finally:
    conn.close()


실행 SQL: CREATE TABLE IF NOT EXISTS "test" ("id" TEXT NOT NULL, "name" TEXT NOT NULL, "descript" TEXT, PRIMARY KEY ("id"))
테이블 생성 완료: test


## 업서트


In [3]:
# 3) test 테이블 업서트
import json

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 반드시 insert_columns와 같은 키를 가져야 합니다.
# TEXT 컬럼은 dict/list를 넣으면 자동으로 JSON 문자열로 변환합니다.
ROWS = [
    {
        'id': 'GOLDEN',
        'name': 'GOLDEN 성격유형검사',
        'descript': '성향을 기반으로 16가지 성격유형을 정의하는 검사',
    },
    {
        'id': 'STS',
        'name': 'STS 6요인 기질검사',
        'descript': '영유아/아동/성인용 대상으로 기질검사',
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'"{c}"' for c in insert_columns])

# insert 후 데이터 수정하고 싶을 때
# 기존 데이터를 덮어쓰도록 ON CONFLICT 절 추가 (sqlite)
conflict_columns = ['id']
update_columns = [c for c in insert_columns if c not in conflict_columns]
conflict_sql = ', '.join([f'"{c}"' for c in conflict_columns])
update_sql = ', '.join([f'"{c}" = excluded."{c}"' for c in update_columns])
insert_sql = (
    f'INSERT INTO "{TABLE_NAME}" ({col_sql}) VALUES ({placeholders}) '
    f'ON CONFLICT ({conflict_sql}) DO UPDATE SET {update_sql}'
)
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

print('실행 SQL:', insert_sql)
print('입력 건수:', len(values))

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'업서트 완료: {len(values)}건')
finally:
    conn.close()


입력 대상 컬럼: ['id', 'name', 'descript']
실행 SQL: INSERT INTO "test" ("id", "name", "descript") VALUES (?, ?, ?) ON CONFLICT ("id") DO UPDATE SET "name" = excluded."name", "descript" = excluded."descript"
입력 건수: 2
업서트 완료: 2건


In [ ]:
# 4) 결과 조회
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute(f'SELECT * FROM "{TABLE_NAME}" ORDER BY "id"')
    rows = cur.fetchall()
    print(f'총 {len(rows)}건')
    for row in rows:
        print(row)
finally:
    conn.close()

총 2건
('GOLDEN', 'GOLDEN 성격유형검사', '성향을 기반으로 16가지 성격유형을 정의하는 검사')
('STS', 'STS 6요인 기질검사', '영유아/아동/성인용 대상으로 기질검사')
